# Evaluating ranked retrieval

Retrieval quality is measured against external relevance judgments. Precision@k emphasizes purity, Recall@k measures coverage, reciprocal rank rewards an early first hit, MAP averages precision at every hit and nDCG supports graded relevance.

In [2]:
import numpy as np
import pandas as pd
from nlp_portfolio.retrieval_eval import (precision_at_k, recall_at_k, reciprocal_rank,
                                          average_precision, ndcg_at_k, evaluate_run)

ranking = ["d3", "d1", "d4", "d2", "d5"]
grades = {"d1": 3, "d2": 2, "d3": 0, "d4": 1, "d5": 0}
relevant = {key for key, value in grades.items() if value > 0}
pd.Series({
    "P@3": precision_at_k(ranking, relevant, 3),
    "R@3": recall_at_k(ranking, relevant, 3),
    "RR": reciprocal_rank(ranking, relevant),
    "AP": average_precision(ranking, relevant),
    "nDCG@3": ndcg_at_k(ranking, grades, 3),
}).round(4)

P@3       0.6667
R@3       0.6667
RR        0.5000
AP        0.6389
nDCG@3    0.5234
dtype: float64

nDCG uses gain (2^{rel}-1) and logarithmic discount. It distinguishes a highly relevant document from a marginally relevant one, while binary MAP treats both as hits.

In [4]:
qrels = {
    "q1": {"d1", "d2"},
    "q2": {"d3"},
    "q3": {"d2", "d5"},
}
baseline = {
    "q1": ["d4", "d1", "d2"],
    "q2": ["d3", "d2", "d1"],
    "q3": ["d1", "d2", "d4"],
}
candidate = {
    "q1": ["d1", "d2", "d4"],
    "q2": ["d4", "d3", "d2"],
    "q3": ["d5", "d2", "d1"],
}
pd.DataFrame({
    "baseline": evaluate_run(baseline, qrels, k=3),
    "candidate": evaluate_run(candidate, qrels, k=3),
})

,baseline,candidate
precision,0.444444,0.555556
recall,0.833333,1.000000
rr,0.666667,0.833333
ap,0.611111,0.833333


## Paired uncertainty over queries

Metrics from the same queries are paired. Resampling query identifiers preserves that pairing and estimates uncertainty in the system-level difference.

In [6]:
from nlp_portfolio.evaluation import paired_bootstrap_delta

baseline_rr = np.array([reciprocal_rank(baseline[q], qrels[q]) for q in qrels])
candidate_rr = np.array([reciprocal_rank(candidate[q], qrels[q]) for q in qrels])
low, high = paired_bootstrap_delta(baseline_rr, candidate_rr, samples=5000, seed=42)
print("mean MRR difference:", round(np.mean(candidate_rr - baseline_rr), 3))
print("paired bootstrap interval:", (round(low, 3), round(high, 3)))

mean MRR difference: 0.167
paired bootstrap interval: (-0.5, 0.5)


Three queries are sufficient to demonstrate the calculation and insufficient for a reliable product decision. Report the number of queries, annotation protocol, pooling procedure and confidence interval. Never evaluate a retriever using its own similarity score as relevance.